# The Data-Driven Social Engagement Initiative

## Downloading and structuring of data

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("datasnaek/youtube")

print("Path to dataset files:", path)

c:\Users\ADITYA\Desktop\New folder (2)\major project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 55.9M/55.9M [00:06<00:00, 9.07MB/s]

Extracting files...


Path to dataset files: C:\Users\ADITYA\.cache\kagglehub\datasets\datasnaek\youtube\versions\24


In [2]:
import re
import numpy as np
import pandas as pd

# Helper: Convert arbitrary string headers to clean snake_case
def to_snake_case(col_name: str) -> str:
    col_name = col_name.strip()
    col_name = re.sub(r'[\s\-]+', '_', col_name)
    col_name = re.sub(r'([a-z\d])([A-Z])', r'\1_\2', col_name)
    col_name = re.sub(r'[^\w]', '', col_name)
    return col_name.lower()

# ==========================================
# 1. Clean Videos Dataset
# ==========================================
print("Loading and cleaning Videos dataset...")
df_videos = pd.read_csv(
    path + "/USvideos.csv", # Update filename if different
    on_bad_lines='skip',
    low_memory=False
)

# Standardize column headers
df_videos.columns = [to_snake_case(c) for c in df_videos.columns]

# Some copies of this dataset use different column names -> map them to the names used below
rename_map = {'comment_count': 'comment_total', 'trending_date': 'date'}
df_videos = df_videos.rename(columns={k: v for k, v in rename_map.items()
                                      if k in df_videos.columns and v not in df_videos.columns})

# Drop rows missing primary identifiers
df_videos = df_videos.dropna(subset=['video_id', 'title']).copy()

# Enforce explicit numeric data types
numeric_video_cols = ['category_id', 'views', 'likes', 'dislikes', 'comment_total']
for col in numeric_video_cols:
    if col in df_videos.columns:
        df_videos[col] = pd.to_numeric(df_videos[col], errors='coerce').fillna(0).astype(np.int64)


if 'date' in df_videos.columns:
    day_num = pd.to_numeric(df_videos['date'], errors='coerce')
    if day_num.notna().mean() < 0.9:  # full date strings like "17.14.11" -> convert to day offsets
        parsed = pd.to_datetime(df_videos['date'], format='%y.%d.%m', errors='coerce')
        day_num = (parsed - parsed.min()).dt.days
    df_videos['date'] = day_num.fillna(0).astype(np.int64)

# Clean text columns (fill nulls and strip whitespace)
text_video_cols = ['title', 'channel_title', 'tags', 'thumbnail_link', 'video_id']
for col in text_video_cols:
    if col in df_videos.columns:
        df_videos[col] = df_videos[col].fillna('').astype(str).str.strip()


rows_before = len(df_videos)
df_videos = df_videos[df_videos['views'] > 0].copy()
print(f"Dropped {rows_before - len(df_videos):,} rows with 0 views")

print(f"Cleaned Videos rows: {len(df_videos):,}")

# ==========================================
# 3. Verification & Integrity Checks
# ==========================================
print("\n--- Videos Dataset Summary ---")
print(df_videos.info())
print(df_videos.describe())
print("\nMissing values in Videos:")
print(df_videos.isnull().sum())

Loading and cleaning Videos dataset...
Dropped 10 rows with 0 views
Cleaned Videos rows: 7,982

--- Videos Dataset Summary ---
<class 'pandas.DataFrame'>
Index: 7982 entries, 0 to 7991
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   video_id        7982 non-null   str  
 1   title           7982 non-null   str  
 2   channel_title   7982 non-null   str  
 3   category_id     7982 non-null   int64
 4   tags            7982 non-null   str  
 5   views           7982 non-null   int64
 6   likes           7982 non-null   int64
 7   dislikes        7982 non-null   int64
 8   comment_total   7982 non-null   int64
 9   thumbnail_link  7982 non-null   str  
 10  date            7982 non-null   int64
dtypes: int64(6), str(5)
memory usage: 3.3 MB
None
       category_id         views         likes       dislikes  comment_total  \
count  7982.000000  7.982000e+03  7.982000e+03    7982.000000    7982.000000   
mean     20.

In [3]:
# ==========================================
# 2. Clean Comments Dataset
# ==========================================
print("\nLoading and cleaning Comments dataset...")
df_comments = pd.read_csv(
    path + "/UScomments.csv", # Update filename if different
    on_bad_lines='skip',
    low_memory=False
)

# Standardize column headers
df_comments.columns = [to_snake_case(c) for c in df_comments.columns]

# Drop rows missing comment text or video ID
df_comments = df_comments.dropna(subset=['video_id', 'comment_text']).copy()

# Text sanitization
df_comments['video_id'] = df_comments['video_id'].astype(str).str.strip()
df_comments['comment_text'] = df_comments['comment_text'].astype(str).str.strip()

# Drop rows where comment_text became empty after whitespace stripping
df_comments = df_comments[df_comments['comment_text'] != ''].copy()

# Coerce numeric fields to handle unescaped text leakage
numeric_comment_cols = ['likes', 'replies']
for col in numeric_comment_cols:
    if col in df_comments.columns:
        df_comments[col] = pd.to_numeric(df_comments[col], errors='coerce').fillna(0).astype(np.int64)

print(f"Cleaned Comments rows: {len(df_comments):,}")


# ==========================================
# 3. Verification & Integrity Checks
# ==========================================

print("\n--- Comments Dataset Summary ---")
print(df_comments.info())
print(df_comments.describe())
print("\nMissing values in Comments:")
print(df_comments.isnull().sum())


Loading and cleaning Comments dataset...
Cleaned Comments rows: 691,372

--- Comments Dataset Summary ---
<class 'pandas.DataFrame'>
Index: 691372 entries, 0 to 691399
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   video_id      691372 non-null  str  
 1   comment_text  691372 non-null  str  
 2   likes         691372 non-null  int64
 3   replies       691372 non-null  int64
dtypes: int64(2), str(2)
memory usage: 90.0 MB
None
               likes        replies
count  691372.000000  691372.000000
mean        5.821189       0.425544
std       243.197389       9.923353
min         0.000000       0.000000
25%         0.000000       0.000000
50%         0.000000       0.000000
75%         0.000000       0.000000
max     48772.000000     529.000000

Missing values in Comments:
video_id        0
comment_text    0
likes           0
replies         0
dtype: int64


### How the Hype features are built (what changed and why)

* Each video appears once per day it was on the trending page, so we sort the snapshots **by day** (not by views) to find the true first and last snapshot.
* **Hype Velocity** is now the views a video had on its **first day on the trending page** (its day-one momentum). The old version (views gained / days trending) was `0` for every video that trended only one day, and for the others it was measured over the same period as the engagement outcome.
* **Engagement Rate** = (likes + comments) / views, taken from the video's **last** snapshot.

In [4]:
# 1. Sort snapshots in time order so "first" and "last" really mean first and last day trending
df_videos = df_videos.sort_values(by=['video_id', 'date'], ascending=[True, True])

# 2. Calculate the Hype metrics by grouping all rows of the same video
hype_df = df_videos.groupby('video_id').agg(
    initial_views=('views', 'first'),   # views on the FIRST day it trended
    final_views=('views', 'last'),      # views on the LAST day it trended
    days_trending=('video_id', 'count') # Each row represents a day on the trending page
).reset_index()

# 3. Engineer the specific Hype features
hype_df['views_gained'] = (hype_df['final_views'] - hype_df['initial_views']).clip(lower=0)

# Day-one momentum: how many views the video already had when it first hit the trending page.
# (days_trending / views_gained are kept for the EDA only - they describe what happened AFTER launch,
#  so they are NOT used as predictors in Notebook 3)
hype_df['hype_velocity_per_day'] = hype_df['initial_views']

# 4. Now, deduplicate the main videos dataframe (keep the latest stats)
df_videos_unique = df_videos.drop_duplicates(subset=['video_id'], keep='last').copy()

# 5. Merge the new Hype features into the unique videos dataframe
df_videos_final = pd.merge(df_videos_unique, hype_df.drop(columns=['final_views']), on='video_id', how='left')

# Calculate the final Engagement Rate using the last snapshot's stats
df_videos_final['final_views'] = df_videos_final['views']
df_videos_final['engagement_rate'] = (df_videos_final['likes'] + df_videos_final['comment_total']) / df_videos_final['final_views']
df_videos_final = df_videos_final.replace([np.inf, -np.inf], np.nan).dropna(subset=['engagement_rate'])

print("Hype Feature Engineering Complete.")
print(f"Unique videos: {len(df_videos_final):,} | videos seen on only 1 day: {(df_videos_final['days_trending'] == 1).mean():.1%}")
display(df_videos_final[['video_id', 'final_views', 'days_trending', 'hype_velocity_per_day', 'engagement_rate']].head())

Hype Feature Engineering Complete.
Unique videos: 2,362 | videos seen on only 1 day: 15.1%


,video_id,final_views,days_trending,hype_velocity_per_day,engagement_rate
0,--JinobXWPk,1319945,1,1319945,0.034636
1,-1fzGnFwz9M,189414,1,189414,0.038846
2,-3AGlBYyLjo,2143,1,2143,0.009333
3,-3lMEZ6k5NA,201901,1,201901,0.052625
4,-5sCWsLlTCI,85052,1,85052,0.018694


In [5]:
# Save clean intermediate files to Colab's active runtime disk
df_videos_final.to_csv("cleaned_videos_v1.csv", index=False)
df_comments.to_csv("cleaned_comments_v1.csv", index=False)
print("Files saved to Colab local runtime: cleaned_videos_v1.csv & cleaned_comments_v1.csv")

Files saved to Colab local runtime: cleaned_videos_v1.csv & cleaned_comments_v1.csv


## Audience Sentiment Analyzer

In [6]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download the VADER dictionary
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ADITYA\AppData\Roaming\nltk_data...


In [7]:
# Load the comments we cleaned in the first step
df_comments = pd.read_csv("cleaned_comments_v1.csv")

print(df_comments.shape)
df_comments.head(3)

(691372, 4)


,video_id,comment_text,likes,replies
0,XpVt6Z1Gjjo,Logan Paul it's yo big day ‼️‼️‼️,4,0
1,XpVt6Z1Gjjo,I've been following you from the start of your...,3,0
2,XpVt6Z1Gjjo,Say hi to Kong and maverick for me,3,0


In [8]:
# Let's see how VADER scores a single comment
sample_comment = df_comments.iloc[0]['comment_text']
print(sample_comment)

# Get polarity scores
scores = sia.polarity_scores(sample_comment)
print(scores)

Logan Paul it's yo big day ‼️‼️‼️
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


In [9]:
# Function to fulfill Deliverable 3: Tag as "Relatable" or "Neutral"
def get_sentiment_tag(text):
    # Extract just the compound score
    score = sia.polarity_scores(str(text))['compound']

    # 0.4 is a standard threshold for strong positive emotion
    if score >= 0.4:
        return "Relatable"
    else:
        return "Neutral"

# Test it on our sample
print(get_sentiment_tag(sample_comment))

Neutral


In [10]:
df_comments['sentiment_tag'] = df_comments['comment_text'].apply(get_sentiment_tag)

print(df_comments['sentiment_tag'].value_counts())

sentiment_tag
Neutral      447217
Relatable    244155
Name: count, dtype: int64


In [11]:
# Convert tags to 1s and 0s for math
df_comments['is_relatable'] = (df_comments['sentiment_tag'] == 'Relatable').astype(int)

# Group by video_id to calculate the percentage of comments that are relatable
sentiment_summary = df_comments.groupby('video_id').agg(
    total_comments_scored=('comment_text', 'count'),
    relatability_ratio=('is_relatable', 'mean') # The mean of 1s and 0s gives the exact percentage
).reset_index()

# Keep only videos with enough scored comments for the percentage to be meaningful.
MIN_COMMENTS = 20
before = len(sentiment_summary)
sentiment_summary = sentiment_summary[sentiment_summary['total_comments_scored'] >= MIN_COMMENTS].copy()
print(f"Videos with >= {MIN_COMMENTS} scored comments: {len(sentiment_summary):,} (dropped {before - len(sentiment_summary):,})")

sentiment_summary.head()

Videos with >= 20 scored comments: 2,144 (dropped 122)


,video_id,total_comments_scored,relatability_ratio
0,--JinobXWPk,100,0.190000
1,-1fzGnFwz9M,100,0.390000
3,-5sCWsLlTCI,67,0.358209
4,-6Zc8Co2H3w,400,0.397500
5,-AJyaVduxCc,291,0.312715


In [12]:
# Load cleaned videos and merge the sentiment scores back in
df_videos = pd.read_csv("cleaned_videos_v1.csv")

# Create the final unified table
master_df = pd.merge(df_videos, sentiment_summary, on='video_id', how='inner')

print(master_df.shape)

# Save this! This is the file you will download for your Dashboard.
master_df.to_csv("master_dataset.csv", index=False)

# Check the final output
master_df[['video_id', 'views', 'relatability_ratio']].head()

(2143, 19)


,video_id,views,relatability_ratio
0,--JinobXWPk,1319945,0.190000
1,-1fzGnFwz9M,189414,0.390000
2,-5sCWsLlTCI,85052,0.358209
3,-6Zc8Co2H3w,1870380,0.397500
4,-AJyaVduxCc,334715,0.312715


In [13]:
master_df.head()

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date,initial_views,days_trending,views_gained,hype_velocity_per_day,final_views,engagement_rate,total_comments_scored,relatability_ratio
0,--JinobXWPk,DANGEROUS Jungle Spider!,Brave Wilderness,15,adventure|adventurous|animals|breaking|breakin...,1319945,38949,533,6768,https://i.ytimg.com/vi/--JinobXWPk/default.jpg,20,1319945,1,0,1319945,1319945,0.034636,100,0.190000
1,-1fzGnFwz9M,9 Things You Need To Know About Kittens - Simo...,Simon's Cat,15,cartoon|simons cat|simon's cat|simonscat|simon...,189414,7070,112,288,https://i.ytimg.com/vi/-1fzGnFwz9M/default.jpg,13,189414,1,0,189414,189414,0.038846,100,0.390000
2,-5sCWsLlTCI,SNL Host Kumail Nanjiani and P!nk Share Favori...,Saturday Night Live,24,saturday night live|snl|snl season 43|kumail n...,85052,1458,97,132,https://i.ytimg.com/vi/-5sCWsLlTCI/default.jpg,14,85052,1,0,85052,85052,0.018694,67,0.358209
3,-6Zc8Co2H3w,Honest Trailers - Star Trek: The Next Generation,Screen Junkies,1,screen junkies|screenjunkies|sj news|honest tr...,1870380,48211,1243,6526,https://i.ytimg.com/vi/-6Zc8Co2H3w/default.jpg,30,1956813,4,0,1956813,1870380,0.029265,400,0.397500
4,-AJyaVduxCc,Why Is Ayesha Curry Sending Pics of Her Feet t...,The Real Daytime,24,the real|daytime|talk show|women|tamera mowry|...,334715,1742,62,279,https://i.ytimg.com/vi/-AJyaVduxCc/default.jpg,20,313279,3,21436,313279,334715,0.006038,291,0.312715
